# Batch Acoustic Feature Extraction
Loops over **every** aligned month in `master_df_clean.parquet`, extracts an expanded acoustic feature set for each `.wav` clip, aggregates clips up to one row per 3-minute rainfall window, and writes one `features_<month_name>.parquet` per dataset to `/kaggle/working/`.

Handles both audio schemes noted in the audit report (17-18 clips of 10s, or 57-60 clips of 3s) transparently, since aggregation just runs over however many clips are present in a window.

In [ ]:
import os
import re
import gc
import glob
import warnings

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm import tqdm

warnings.filterwarnings("ignore")

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Load the cleaned, consolidated label dataframe (already deduplicated and
# stripped of the invalid >100mm sensor-overflow rows, per audit_report_v1).
master_df = pd.read_parquet(
    "/kaggle/input/datasets/amarnathdj/audit-report-v1/master_df_clean.parquet"
)

print(master_df.shape)
master_df.head()


In [ ]:
all_source_pickles = master_df["source_pickle"].unique()

print(f"Found {len(all_source_pickles)} datasets to process:")
for s in all_source_pickles:
    n = (master_df["source_pickle"] == s).sum()
    print(f"  - {s:45s} ({n} windows)")


## Feature extraction (per audio clip)
Expanded relative to the original single-month version:
- Time-domain: zero-crossing rate, RMS (mean/std/max instead of just mean, since rain intensity shows up as energy *variability* within a clip, not just its average)
- Spectral shape: centroid, bandwidth, rolloff, **spectral flatness** (new — separates noise-like rain sound from tonal background noise), **spectral contrast** across 7 bands (new)
- **Chroma** (new — helps the model discount non-rain tonal/harmonic sounds, e.g. traffic, voices)
- MFCCs: still 13 coefficients (mean/std), plus **delta-MFCCs** (new — first-order time derivative, captures the *texture change* of the sound rather than a static snapshot)

In [ ]:
N_MFCC = 13

def extract_clip_features(path):
    """Extract an acoustic feature dict from a single .wav clip."""
    y, sr = sf.read(path)

    # Convert stereo to mono if necessary
    if y.ndim > 1:
        y = np.mean(y, axis=1)

    y = y.astype(np.float32)

    # Guard against very short/corrupt clips that break spectral feature windows
    min_len = 2048
    if len(y) < min_len:
        y = np.pad(y, (0, min_len - len(y)))

    features = {}

    # --- Time-domain ---
    features["zcr"] = librosa.feature.zero_crossing_rate(y).mean()

    rms = librosa.feature.rms(y=y)
    features["rms_mean"] = rms.mean()
    features["rms_std"] = rms.std()
    features["rms_max"] = rms.max()

    # --- Spectral shape ---
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    features["centroid_mean"] = centroid.mean()
    features["centroid_std"] = centroid.std()

    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features["bandwidth_mean"] = bandwidth.mean()

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features["rolloff_mean"] = rolloff.mean()

    flatness = librosa.feature.spectral_flatness(y=y)
    features["flatness_mean"] = flatness.mean()

    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        features[f"contrast_{i+1}_mean"] = contrast[i].mean()

    # --- Chroma ---
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features["chroma_mean"] = chroma.mean()

    # --- MFCCs + delta ---
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    mfcc_delta = librosa.feature.delta(mfcc)

    for i in range(N_MFCC):
        features[f"mfcc_{i+1}_mean"] = mfcc[i].mean()
        features[f"mfcc_{i+1}_std"] = mfcc[i].std()
        features[f"mfcc_delta_{i+1}_mean"] = mfcc_delta[i].mean()

    return features


In [ ]:
# Quick sanity check on one file before running the full batch
sample_row = master_df.iloc[0]
sample_path = sample_row["wav_files"][0]

print(sample_path)
print(os.path.exists(sample_path))

feat = extract_clip_features(sample_path)
print(f"{len(feat)} features per clip")
feat


## Per-window aggregation
Instead of only averaging clip-level features across a window (which is what the original notebook did), this aggregates with **both mean and std** across clips. The std across clips carries real signal here: a window with bursty/variable rain sound across its clips looks different from one with steady drizzle, even if the mean is similar.

In [ ]:
def aggregate_window(clip_features_list):
    feat_df = pd.DataFrame(clip_features_list)
    agg = {}
    for col in feat_df.columns:
        agg[f"{col}_mean"] = feat_df[col].mean()
        agg[f"{col}_std"] = feat_df[col].std()
    agg["n_clips_used"] = len(feat_df)
    return agg


## Batch processing — one `features_<month_name>.parquet` per dataset
Iterates over every distinct `source_pickle` in `master_df`, extracts + aggregates features for that month only, and writes it out immediately (rather than holding everything in memory at once, which won't scale across ~2.5 years of data).

In [ ]:
def month_name_from_pickle(source_pickle):
    return re.sub(r"_aligned_dataset\.pkl$", "", source_pickle)


def process_dataset(df_subset):
    features = []

    for _, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
        clip_features = []

        for path in row["wav_files"]:
            try:
                feat = extract_clip_features(path)
                clip_features.append(feat)
            except Exception:
                continue

        if len(clip_features) == 0:
            continue

        sample = aggregate_window(clip_features)
        sample["rainfall_mm"] = row["rainfall_mm"]
        sample["timestamp"] = row["timestamp"]
        sample["wav_count"] = row["wav_count"]

        features.append(sample)

    gc.collect()
    return pd.DataFrame(features)


In [ ]:
summary = []

for source in all_source_pickles:
    month_name = month_name_from_pickle(source)
    print(f"\n=== Processing {month_name} ===")

    df_subset = master_df[master_df["source_pickle"] == source].copy()
    print(f"  windows to process: {len(df_subset)}")

    feature_df = process_dataset(df_subset)

    if feature_df.empty:
        print(f"  WARNING: no features extracted for {month_name}, skipping save.")
        continue

    # Classification target
    feature_df["rain"] = (feature_df["rainfall_mm"] > 0).astype(int)

    out_path = f"{OUTPUT_DIR}/features_{month_name}.parquet"
    feature_df.to_parquet(out_path, index=False)

    print(f"  saved -> {out_path}  shape={feature_df.shape}")

    summary.append({
        "month": month_name,
        "n_windows": len(feature_df),
        "n_features": feature_df.shape[1],
        "n_rain_windows": int(feature_df["rain"].sum()),
        "output_path": out_path,
    })

    del feature_df
    gc.collect()

summary_df = pd.DataFrame(summary)
summary_df


## Processing Status
Check what's been extracted so far.

In [ ]:
import glob
import os

feature_files = sorted(glob.glob(f"{OUTPUT_DIR}/features_*.parquet"))
feature_files = [f for f in feature_files if 'combined' not in f]

print(f"✓ {len(feature_files)} monthly feature files on disk:")
for f in feature_files:
    df = pd.read_parquet(f)
    month = os.path.basename(f).replace('features_', '').replace('.parquet', '')
    rain_pct = 100 * df['rain'].sum() / len(df)
    print(f"  {month:20s} {len(df):5d} windows ({rain_pct:5.1f}% rain)")

total_windows = sum([len(pd.read_parquet(f)) for f in feature_files])
print(f"\nTotal: {total_windows} windows across all months")

### Batch Processing Workflow

Your data processing is **incremental and safe**:

1. **Run 1**: Upload datasets A-E → run extraction → creates `features_A.parquet` ... `features_E.parquet`
2. **Run 2**: Delete A-E from Kaggle inputs, upload F-H → run extraction → creates `features_F.parquet` ... `features_H.parquet`
3. **Combine**: The cell below reads **all** `features_*.parquet` files currently in `/kaggle/working/`, so it automatically includes A-H (even though A-E are no longer in the input datasets).

**Each monthly `.parquet` is immutable once written**. The `features_all_combined.parquet` is rebuilt from scratch each time you run the combine cell, so it always reflects everything that's been extracted so far.

## Optional: combine all monthly parquet files into one
Handy for the actual model training step, while still keeping the per-month files for auditing or targeted re-processing of a single month.

In [ ]:
all_feature_files = sorted(glob.glob(f"{OUTPUT_DIR}/features_*.parquet"))
print(f"Found {len(all_feature_files)} monthly feature files")

combined_df = pd.concat(
    [pd.read_parquet(f) for f in all_feature_files if "combined" not in f],
    ignore_index=True
)

combined_out_path = f"{OUTPUT_DIR}/features_all_combined.parquet"
combined_df.to_parquet(combined_out_path, index=False)

print(f"Combined shape: {combined_df.shape}")
print(f"Saved -> {combined_out_path}")
